# Étape 1 — Extraction de patterns

On transforme les valeurs brutes en patterns (préfixes, premiers mots).

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
from src.pattern_extractor import extract_prefixes, extract_tokens, prefix_patterns, token_patterns, extract_all_patterns

## Test sur des valeurs individuelles

In [2]:
# préfixes
print(extract_prefixes('60601', 3))   
print(extract_prefixes('90012', 5))   
print(extract_prefixes(None, 3))      
print(extract_prefixes('', 3))        

['6', '60', '606']
['9', '90', '900', '9001', '90012']
[]
[]


In [3]:
# tokens
print(extract_tokens('John Smith'))      
print(extract_tokens('Aarhus, Pam J.')) 

['John', 'Smith']
['Aarhus', 'Pam', 'J']


## Test sur une colonne entière — t2.csv

In [4]:
t2 = pd.read_csv('../data/pfd_validation/t2.csv')

# préfixes de ZIP avec longueur max 3, support >= 30
zip_patterns = prefix_patterns(t2['ZIP'], k_max=3, min_support=30)
print(f'{len(zip_patterns)} patterns de ZIP trouvés')
# afficher les 10 plus fréquents
for p, idxs in sorted(zip_patterns.items(), key=lambda x: -len(x[1]))[:10]:
    print(f'  "{p}" -> {len(idxs)} lignes')

30 patterns de ZIP trouvés
  "6" -> 2509 lignes
  "60" -> 2456 lignes
  "606" -> 2131 lignes
  "1" -> 354 lignes
  "10" -> 285 lignes
  "100" -> 242 lignes
  "2" -> 136 lignes
  "600" -> 97 lignes
  "0" -> 96 lignes
  "4" -> 88 lignes


In [5]:
# premiers mots de NAME
name_tokens = token_patterns(t2['NAME'], min_support=5)
print(f'{len(name_tokens)} premiers mots trouvés')
for t, idxs in sorted(name_tokens.items(), key=lambda x: -len(x[1]))[:10]:
    print(f'  "{t}" -> {len(idxs)} lignes')

224 premiers mots trouvés
  "Illinois" -> 64 lignes
  "Morgan" -> 53 lignes
  "Chicago" -> 51 lignes
  "SEIU" -> 45 lignes
  "The" -> 43 lignes
  "William" -> 40 lignes
  "Neal" -> 37 lignes
  "DLA" -> 33 lignes
  "KPMG" -> 33 lignes
  "J" -> 33 lignes


## extract_all_patterns 

In [6]:
t2['ZIP'].value_counts()

ZIP
60606      412
60601      409
60603      279
60602      181
60654      126
          ... 
78701        1
06901        1
63102        1
10105        1
H9P 2S4      1
Name: count, Length: 358, dtype: int64

In [7]:
# sur la colonne ZIP
all_p = extract_all_patterns(t2['ZIP'], k_max=3, min_support=30)
for match_type, pmap in all_p.items():
    print(f'{match_type}: {len(pmap)} patterns')

startswith: 30 patterns
first_token: 19 patterns
